# Lab: guidance, samplers and how to measure them

**Session 7: Diffusion and multimodal generation**
Goal: turn every knob of Stable Diffusion that the lecture explained, look at the effect, and measure it with CLIP score and FID.


## Setup

Runs on **Google Colab with a GPU**: _Runtime → Change runtime type → T4 GPU_. The model (Stable Diffusion 1.5, half precision) is a ~2 GB download, about 1–2 minutes. One 512×512 image with 20 steps takes ~2–3 seconds on a T4.

Running example, as in the lecture: _"an ECB tower at sunset, watercolor"_.


In [ ]:
%pip install -q diffusers transformers accelerate "torchmetrics[image]" ipywidgets

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from diffusers import (
    DDIMScheduler,
    DPMSolverMultistepScheduler,
    EulerDiscreteScheduler,
    StableDiffusionPipeline,
)

if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda", torch.float16
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", torch.float16
else:
    raise SystemExit(
        "No GPU found. On Colab: Runtime > Change runtime type > T4 GPU, then run again."
    )

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=DTYPE, variant="fp16"
).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

SAMPLERS = {
    "DDIM": DDIMScheduler,
    "Euler": EulerDiscreteScheduler,
    "DPM++ 2M": DPMSolverMultistepScheduler,
}
PROMPT = ...  # TODO: COMPLETE
print("ready on", DEVICE)

In [ ]:
def generate(
    prompt=PROMPT,
    guidance=7.5,
    steps=20,
    sampler="Euler",
    seed=0,
    negative="",
    **kwargs
):
    """One image, fully determined by its settings. Returns (image, seconds)."""
    pipe.scheduler = SAMPLERS[sampler].from_config(pipe.scheduler.config)
    generator = torch.Generator(DEVICE).manual_seed(seed)
    start = time.perf_counter()
    image = pipe(
        prompt,
        guidance_scale=guidance,
        num_inference_steps=steps,
        generator=generator,
        negative_prompt=negative or None,
        **kwargs
    ).images[0]
    return image, time.perf_counter() - start


def show_row(images, titles, size=3.2):
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for ax, img, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Station 1: Playground

Change one setting at a time and press **Run Interact**. Try:

- guidance 1, then 7.5, then 20
- 5 steps, then 50
- the same settings with seed 0, then seed 1
- negative prompt: `blurry, low quality, people`


In [ ]:
import ipywidgets as widgets
from IPython.display import display


def playground(prompt, guidance, steps, sampler, seed, negative):
    image, seconds = generate(prompt, guidance, steps, sampler, seed, negative)
    print(f"{seconds:.1f} s")
    display(image.resize((384, 384)))


widgets.interact_manual(
    playground,
    prompt=widgets.Text(PROMPT, layout=widgets.Layout(width="500px")),
    guidance=widgets.FloatSlider(value=7.5, min=1, max=20, step=0.5),
    steps=widgets.IntSlider(value=20, min=2, max=50),
    sampler=widgets.Dropdown(options=list(SAMPLERS), value="Euler"),
    seed=widgets.IntText(0),
    negative=widgets.Text(""),
)

## Station 2: Watch it denoise

The pipeline starts from pure noise in the **latent space** (4 × 64 × 64) and removes a little noise at each step. A latent is not an image: to look at it, it must go through the **VAE decoder**.

Complete `keep_frames`, called by the pipeline at the end of every step. Every `every` steps, decode the current latent and store the image in `frames`:

1. `latents = callback_kwargs["latents"]`
2. decode: `pipe.vae.decode(latents / pipe.vae.config.scaling_factor).sample`
3. turn it into a picture: `pipe.image_processor.postprocess(decoded, output_type="pil")[0]`
4. return `callback_kwargs` unchanged


In [ ]:
frames = []


def keep_frames(pipeline, step, timestep, callback_kwargs, every=4):
    if (
        ...
    ):  # TODO: every `every` steps, decode callback_kwargs["latents"] with the VAE and append the picture to frames
        latents = callback_kwargs["latents"]
        with torch.no_grad():
            decoded = pipe.vae.decode(latents / pipe.vae.config.scaling_factor).sample
        frames.append(pipe.image_processor.postprocess(decoded, output_type="pil")[0])
    return callback_kwargs


image, _ = generate(
    PROMPT,
    steps=20,
    seed=0,
    callback_on_step_end=keep_frames,
    callback_on_step_end_tensor_inputs=["latents"],
)
assert len(frames) == 5, f"expected 5 frames (steps 0, 4, 8, 12, 16), got {len(frames)}"
show_row(
    frames + [image],
    [f"step {4 * i}" for i in range(len(frames))] + ["final"],
    size=2.6,
)

**Question:** at which step can you first recognise the tower? What appears first: the layout and colours, or the details?


## Station 3: Guidance sweep, scored with CLIP

Same prompt, same seed, only the **guidance scale** $w$ changes.

**CLIP score** = 100 × cosine(CLIP image embedding, CLIP text embedding), floored at 0. Higher = the image matches the prompt better. Complete `clip_score`; the test compares it with `torchmetrics`.


In [ ]:
from transformers import CLIPModel, CLIPProcessor
import torch.nn.functional as F

CLIP_NAME = "openai/clip-vit-base-patch16"
clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)

In [ ]:
def clip_score(image, prompt):
    inputs = clip_processor(
        text=[prompt], images=image, return_tensors="pt", padding=True
    ).to(DEVICE)
    with torch.no_grad():
        image_out = clip_model.get_image_features(pixel_values=inputs["pixel_values"])
        text_out = clip_model.get_text_features(
            input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"]
        )

    image_emb = (
        image_out.pooler_output if hasattr(image_out, "pooler_output") else image_out
    )
    text_emb = (
        text_out.pooler_output if hasattr(text_out, "pooler_output") else text_out
    )

    if not isinstance(image_emb, torch.Tensor) and hasattr(
        image_emb, "last_hidden_state"
    ):
        image_emb = image_emb.last_hidden_state[:, 0]
    if not isinstance(text_emb, torch.Tensor) and hasattr(
        text_emb, "last_hidden_state"
    ):
        text_emb = text_emb.last_hidden_state[:, 0]

    image_emb = ...  # TODO: normalize
    text_emb = ...  # TODO: normalize

    cosine = torch.matmul(image_emb, text_emb.T).item()

    return float(max(100 * cosine, 0.0))


test_image, _ = generate(PROMPT, seed=0)
score = clip_score(test_image, PROMPT)
print(f"OK: clip_score = {score:.2f}")

In [ ]:
guidances = [1, 3, 5, 7.5, 12, 20]
row = [generate(PROMPT, guidance=w, seed=0)[0] for w in guidances]
scores = [clip_score(img, PROMPT) for img in row]
show_row(row, [f"w = {w}\nCLIP {s:.1f}" for w, s in zip(guidances, scores)])

plt.plot(guidances, scores, marker="o")
plt.xlabel("guidance scale w")
plt.ylabel("CLIP score")
plt.title(PROMPT)
plt.show()

**Questions:** where does the CLIP score stop improving? What happens to the colours at w = 20? Does the CLIP score see that damage?


## Station 4: Sampler × steps race, then dominance

Same model, same seed, same guidance: only the **sampler** and the **number of steps** change. Each run records its CLIP score and its time.


In [ ]:
runs = []
for sampler in SAMPLERS:
    images = []
    for steps in [5, 10, 25]:
        image, seconds = generate(
            PROMPT, guidance=7.5, steps=steps, sampler=sampler, seed=0
        )
        runs.append(
            {
                "name": f"{sampler}, {steps} steps",
                "clip": clip_score(image, PROMPT),
                "seconds": seconds,
            }
        )
        images.append(image)
    show_row(
        images,
        [f"{r['name']}\nCLIP {r['clip']:.1f}, {r['seconds']:.1f} s" for r in runs[-3:]],
    )

A run is **dominated** if another run is at least as good on both criteria (CLIP score ↑, seconds ↓) and strictly better on at least one. Complete `dominated(a, b)`: does `b` dominate `a`?


In [ ]:
def dominated(a, b):
    """True if run b dominates run a: b["clip"] >= a["clip"], b["seconds"] <= a["seconds"], strictly better on one."""
    # TODO
    raise NotImplementedError


fast_good = {"clip": 32, "seconds": 1.0}
slow_bad = {"clip": 30, "seconds": 3.0}
slow_good = {"clip": 33, "seconds": 3.0}
assert dominated(slow_bad, fast_good) and not dominated(fast_good, slow_bad)
assert not dominated(slow_good, fast_good) and not dominated(
    fast_good, slow_good
)  # a real trade-off
assert not dominated(fast_good, fast_good)  # equal is not better
print("OK: dominated")

In [ ]:
front = [r for r in runs if not any(dominated(r, other) for other in runs)]
for r in runs:
    plt.scatter(r["seconds"], r["clip"], color="#1b8a5a" if r in front else "#8a8984")
    plt.annotate(
        r["name"],
        (r["seconds"], r["clip"]),
        fontsize=8,
        xytext=(4, 4),
        textcoords="offset points",
    )
plt.xlabel("seconds per image (lower is better)")
plt.ylabel("CLIP score (higher is better)")
plt.title("Green: Pareto front. Grey: dominated")
plt.show()
print("Keep:", [r["name"] for r in front])

**Question:** the times here are for one image. What would the front look like for 10,000 product photos? And is a CLIP score from **one** seed enough to drop a sampler?


## Station 5: FID, and why it needs many images

FID compares two **sets** of images: the distance between the means and covariances of their Inception features (lower = more similar). The lecture said it needs thousands of images. Check it.

Below: 24 images at w = 7.5 (set A), 24 more **with the same settings but other seeds** (set B), and 24 at w = 1.5 (set C). Each batch takes about 30 seconds.


In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance

SCENES = [
    "a tower",
    "a bridge",
    "a market square",
    "a harbour",
    "a train station",
    "a library",
    "a park",
    "a skyline",
]
FID_PROMPTS = [f"{scene} at sunset, watercolor" for scene in SCENES] * 3  # 24 prompts


def make_set(guidance, first_seed):
    pipe.scheduler = SAMPLERS["DPM++ 2M"].from_config(pipe.scheduler.config)
    images = []
    for i in range(0, len(FID_PROMPTS), 8):
        gens = [
            torch.Generator(DEVICE).manual_seed(first_seed + i + k) for k in range(8)
        ]
        out = pipe(
            FID_PROMPTS[i : i + 8],
            guidance_scale=guidance,
            num_inference_steps=20,
            generator=gens,
            output_type="np",
        ).images
        images.append(
            torch.from_numpy(out).permute(0, 3, 1, 2)
        )  # (8, 3, H, W) in [0, 1]
    return torch.cat(images)


def fid(set_1, set_2):
    metric = FrechetInceptionDistance(feature=2048, normalize=True)
    metric.update(set_1, real=True)
    metric.update(set_2, real=False)
    return metric.compute().item()


set_a = make_set(7.5, first_seed=0)
set_b = make_set(7.5, first_seed=1000)
set_c = make_set(1.5, first_seed=0)
print(f"FID(A, B), same settings:     {fid(set_a, set_b):.1f}")
print(f"FID(A, C), w = 7.5 vs w = 1.5: {fid(set_a, set_c):.1f}")

**Questions:**

1. A and B come from the **same** model and settings. With enough images their FID would be close to 0. What do you get with 24?
2. Is the gap between FID(A, C) and FID(A, B) large enough to conclude anything?
3. Papers report FID on 10,000 to 50,000 images. Why do they never compare an FID on 24 images with an FID on 5,000?
